In [1]:
clean_up = True # if True, remove all gams related files from working folder before starting
%run stdPackages.ipynb

The file _gams_py_gdb0.gdx is still active and was not deleted.
The file _gams_py_gdb1.gdx is still active and was not deleted.
The file _gams_py_gdb2.gdx is still active and was not deleted.
The file _gams_py_gdb3.gdx is still active and was not deleted.
The file _gams_py_gdb4.gdx is still active and was not deleted.
The file _gams_py_gdb5.gdx is still active and was not deleted.


*Load specific modules used here:*

In [2]:
os.chdir(d['py'])
import mCGE, mGovernment, mOther
from gmsPython import nestingTree
from valueShares import InitTree

# PCOPT

Standard version: No initial wealth, balanced consumption/production in all years, no initial taxes, exogenous labor supply.

## 1. Load, init

In [3]:
name = 'M_PCitory'
nameInput = 'PCitoryDb'
db = GpyDB(os.path.join(d['data'], nameInput), name = f'{name}db', ws = d['work'])
dbIO = db.copy()
ws = db.ws

Initialize model:

In [4]:
M = mCGE.CGE_welfare(name, database = db)

## 1. Add modules

### A. Production

Nesting tree:

In [5]:
nestP = nestingTree.AggTree(name = 'P', trees = {'P': nestingTree.Tree('P', tree = db('nestProduction').to_list(), f = 'CES_scaled')})
nestP(namespace = {str(n)+'_input': n for n in db('n')})

*Get initial values of $\mu$-parameters and intermediate goods:*

In [6]:
v = InitTree(nestP, ws = ws)
v.db['sigma'] = adj.rc_pd(db('sigma'), v.db('s'))
vals = v(dbIO = db)

*Add to main database:*

In [7]:
[db.aom(stdSort(adjMultiIndex.bc(vals[k], db('txE'))), name = k, priority = 'second') for k in ('pD','qD','pS','qS')];
db.aom(vals['mu'], name = 'mu', priority = 'second')
db.aom(gpy(1/vals['mu'], name = 'qNorm', type = 'par'), priority = 'second') # normalizing parameter in 

*Add module:*

In [8]:
M.addProduction('StaticNCES', nestP, partial = False, initFromGms = 'initFirmValueBlock', taxInstr = 'tauS')

### B. Households

Nesting tree:

In [9]:
nest = nestingTree.AggTree(name = 'HH', trees = {'HH': nestingTree.Tree('HH', tree = db('nestHH').to_list(), f = 'CES_scaled')}, ws = ws)
nest(namespace = {str(n)+'_input': n for n in db('n')});

Get starting values:

In [10]:
v = InitTree(nest, ws = ws)
v.db['sigma'] = adj.rc_pd(db('sigma'), v.db('s'))
vals = v(dbIO = db)

*Add pS/qS for the top nest to the qD/pD vectors:*

In [11]:
[vals.__setitem__(f'{k}D', vals[f'{k}D'].combine_first(vals[f'{k}S'])) for k in ('q','p')];

*Add to main database:*

In [12]:
[db.aom(stdSort(adjMultiIndex.bc(vals[k], db('txE'))), name = k, priority = 'second') for k in ('pD','qD')];
db.aom(vals['mu'], name = 'mu', priority = 'second')
db.aom(gpy(1/vals['mu'], name = 'qNorm', type = 'par'), priority = 'second') # normalizing parameter in 

Static version:

In [13]:
nest.db['t0'] = db['t0'] # this is used in the initialization phase - before merging the databases
M.addHousehold('StaticNCES', nest, L2C = db('L2C'), properties = {'taxInstr': 'tauD', 'incInstr': 'jTerm'}, initFromGms = 'init_vU', partial = True)

### C. Government

In [14]:
M.addModule(mGovernment.BalanceLSxGS(name = 'G', database = M.db, sectors = db('s_HH')))
M.m['G'].initStuff(gdx = False)

### D. Welfare module:

In [15]:
M.addWelfareModule('HWPolicyRules','W', active = False) # don't use NLP solver yet.

### E. Inventory

In [16]:
M.db['sInventory'] =pd.Index(['itory'], name = 's')
M.addModule(getattr(mOther, 'InventoryAR')('IVT', M.db, 'sInventory'))
M.m['IVT'].initStuff(gdx = False)
M.db['inventoryAR'].vals = pd.Series(.5, index = M.db('inventoryAR').index, name = 'inventoryAR')

### F. Equilibrium:

*Note: With this being a closed economy, we adjust some sets appropriately here:*

In [17]:
# M.db['nEqui'] = M.db('nEqui').difference(pd.Index(['L'], name = 'n'))
# M.db['d_pEqui'] = M.db('d_pEqui').difference(pd.Index(['L'], name = 'n'))
M.addEquilibriumModule('SmallOpenEq','Equi')

## 2. Calibrate

### A. Small stuff

*Clean up database a bit (this is not necessary, but it removes some variables that are not ultimately used in the model):*

In [18]:
[db.series.__delitem__(k) for k in ('vD','vTax', 'vD_dur','vD_depr','vD_inv', 'vS', 'pD_dur') if k in db.symbols];

For variables that are defined over $t$, but where we do not yet have an initial value for all $t$, extrapolate from data:

*Note: This forces extrapolation of all variables defined over $t$ - if it is important that some variables are not extrapolated, they should be removed from this statement.*

In [19]:
[symbol.__setattr__('vals', extrapolateUpper(symbol.vals, db('tE')[0])) for symbol in [db[k] for k in db.varDom('t')['t']]];

Merge internally, i.e. write a gdx file from the Python database:

In [20]:
db.mergeInternal()

## B. Solve and test

In [21]:
solC = M.solve(state = 'C')

Remove local methods and partial statements - add data from calibration and test baseline solution as well:

In [22]:
[m.__setattr__('initFromGms', None) for m in M.m.values() if hasattr(m, 'initFromGms')]; # remove the GAMS initialization part
[m.__setattr__('partial', False) for m in M.m.values() if hasattr(m, 'partial')]; # remove partial eq. settings for now also.
[M.db.__setitem__(k, solC[k]) for k in M.db.getTypes(['var']) if k in solC.symbols]; # use solution database
M.db.mergeInternal()

Test baseline:

In [23]:
solB = M.solve(state = 'B')
(solB('qD').groupby(['t','n']).sum()-solB('qS').groupby(['t','n']).sum()).dropna().unstack('n')

n,L,agg1,agg2
t,,,
2019,0.0,0.0,0.000000e+00
2020,0.0,0.0,0.000000e+00
2021,0.0,0.0,3.552714e-15
2022,0.0,0.0,-3.552714e-15
2023,0.0,0.0,3.552714e-15
2024,0.0,0.0,0.000000e+00


*After a shock?*

In [24]:
M.db.aom(adj.rc_pd(M.db('mu'), pd.Index(['L'],name='nn'))*1.1, name = 'mu')
M.db.mergeInternal()
test = M.solve(state='B')
(test('qD').groupby(['t','n']).sum()-test('qS').groupby(['t','n']).sum()).dropna().unstack('n')

n,L,agg1,agg2
t,,,
2019,0.0,1.776357e-15,-1.776357e-15
2020,0.0,0.000000e+00,0.000000e+00
2021,0.0,0.000000e+00,0.000000e+00
2022,0.0,1.776357e-15,0.000000e+00
2023,0.0,1.776357e-15,1.776357e-15
2024,0.0,0.000000e+00,0.000000e+00


The effect of changing the wage rate:

In [25]:
M.db.aom(adj.rc_pd(M.db('p'), pd.Index(['L'], name = 'n'))*.8)
M.db.mergeInternal()
test = M.solve(state='B')
(test('qD').groupby(['t','n']).sum()-test('qS').groupby(['t','n']).sum()).dropna().unstack('n')

n,L,agg1,agg2
t,,,
2019,0.0,0.000000e+00,0.000000e+00
2020,0.0,0.000000e+00,1.776357e-15
2021,0.0,1.776357e-15,0.000000e+00
2022,0.0,1.776357e-15,0.000000e+00
2023,0.0,1.776357e-15,1.776357e-15
2024,0.0,0.000000e+00,0.000000e+00


## 3. Optimal taxes:

*Revert to baseline solution:*

In [26]:
[M.db.__setitem__(k, solB[k]) for k in M.db.getTypes(['var']) if k in solB.symbols]; # use solution database

### A. Optimal

*Add policy rule:*

In [27]:
M.m['W'].addRule_addProp('SBts', 'tauS', cond = ('or', [M.g('L', m = 'HH'), M.g('output', m = 'P')]), instrDomains = [['t','s']]);
M.opt = True

Solve:

In [28]:
M.m['W'].initStuff(gdx = True)
solOpt = M.solve()

Optimal to have zero taxes all around:

In [29]:
all(solOpt('tauS') == 0)

True

### B. Optimal after a shock

Solve after technology shock:

In [30]:
M.db.aom(adj.rc_pd(M.db('mu'), M.g('output',m='P'))*1.2)
M.opt = False
M.m['W'].initGroups()
M.db.mergeInternal()
test = M.solve()

Optimal to have zero taxes now?

In [31]:
M.opt = True
M.m['W'].initGroups()
[M.db.__setitem__(k, test[k]) for k in M.db.getTypes(['var']) if k in test.symbols]; # use solution database
M.db.mergeInternal()
testOpt = M.solve()
all(testOpt('tauS') == 0)

True